#   S6E9: Deep Neural Network (Entity Embeddings)

> **Something Completely Different | High Diversity | Keras/TensorFlow**

To achieve the *ultimate* ensemble score, you need models that think differently. Trees (LGBM, XGBoost, CatBoost) make decisions by drawing orthogonal boxes in space. Neural Networks make decisions by drawing smooth, curved boundaries. 

This notebook builds a highly optimized **Deep Neural Network using Entity Embeddings**. Instead of Target Encoding, we pass high-cardinality categorical features (like extracted digits and income brackets) into trainable Embedding Layers. The network learns a dense, multi-dimensional representation of each category!

###   Neural Architecture
- **Entity Embeddings** for Categorical Features (learns spatial relationships between categories)
- **Batch Normalization & Dropout** for extreme regularization
- **Swish Activation** (often outperforms ReLU on tabular data)
- **Cosine Annealing Learning Rate**

---
*Created by [Aman Atar](https://www.kaggle.com/amanatar)*

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings, gc, os
warnings.filterwarnings('ignore')

# Enable mixed precision for speed if running on GPU
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
except:
    pass

# Load competition data
train = pd.read_csv('/kaggle/input/playground-series-s6e9/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e9/test.csv')

TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
test_ids = test['id'].values

print(f"Train: {train.shape}, Test: {test.shape}")

In [ ]:
#                                                    
# FEATURE ENGINEERING FOR NEURAL NETWORKS
#                                                    

train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

# Identify categoricals
cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

print("Extracting digits...")
for c in num_cols:
    for k in [-1, 0, 1, 2]:
        combined[f"{c}_d{k}"] = (combined[c].fillna(0) // (10**k) % 10).astype('int8')

print("Adding CTGAN artifact flags...")
combined['is_30k_spike'] = (combined['Annual_Income_USD'] == 30000.0).astype('int8')
combined['is_millionaire_cliff'] = (combined['Annual_Income_USD'] >= 170537.0).astype('int8')
combined['is_dead_zone'] = ((combined['Annual_Income_USD'] >= 38000.0) & 
                            (combined['Annual_Income_USD'] <= 42000.0)).astype('int8')
combined['is_env_hater'] = (combined['Environmental_Concern_Level'] == 1).astype('int8')

print("Adding smooth keys...")
combined['income_int'] = np.floor(combined['Annual_Income_USD']).astype(str)
combined['income100'] = np.floor(combined['Annual_Income_USD'] / 100.0).astype(str)
combined['income1000'] = np.floor(combined['Annual_Income_USD'] / 1000.0).astype(str)
combined['commute_int'] = np.floor(combined['Daily_Commute_km']).astype(str)
smooth_cats = ['income_int', 'income100', 'income1000', 'commute_int']

# For Neural Networks, we MUST handle missing values and scale numericals
print("Scaling and Encoding...")
embed_cols = cat_cols + smooth_cats
for c in embed_cols:
    combined[c] = combined[c].fillna('Missing').astype(str)
    le = LabelEncoder()
    combined[c] = le.fit_transform(combined[c])

dense_cols = [c for c in combined.columns if c not in embed_cols + ['id', 'is_train', TARGET]]
for c in dense_cols:
    combined[c] = combined[c].fillna(combined[c].median())
    scaler = StandardScaler()
    combined[c] = scaler.fit_transform(combined[[c]])

# Split back
train_df = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
test_df = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET]).reset_index(drop=True)
del combined; gc.collect()

FEATURES_EMBED = embed_cols
FEATURES_DENSE = dense_cols

print(f"Dense Features: {len(FEATURES_DENSE)}")
print(f"Embedding Features: {len(FEATURES_EMBED)}")

In [ ]:
#                                                    
# NEURAL NETWORK ARCHITECTURE
#                                                    

def build_model(train_data):
    inputs = []
    embeddings = []
    
    # 1. Embedding Layers for Categorical Features
    for col in FEATURES_EMBED:
        vocab_size = int(train_data[col].max() + 2)
        # Rule of thumb for embedding dimension
        embed_dim = min(50, int(np.ceil(vocab_size ** 0.25))) 
        
        inp = layers.Input(shape=(1,), name=f'inp_{col}')
        emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inp)
        emb = layers.Flatten()(emb)
        
        inputs.append(inp)
        embeddings.append(emb)
        
    # 2. Dense Layer for Numerical Features
    inp_dense = layers.Input(shape=(len(FEATURES_DENSE),), name='inp_dense')
    inputs.append(inp_dense)
    
    # 3. Concatenate all features
    if embeddings:
        x = layers.Concatenate()(embeddings + [inp_dense])
    else:
        x = inp_dense
        
    # 4. Deep Dense Network
    x = layers.Dense(256, activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(64, activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)
    
    # Output Layer
    output = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    
    model = models.Model(inputs=inputs, outputs=output)
    
    # Optimizer with decay
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    
    return model

# Helper to format data for the model
def format_data(df):
    data = {f'inp_{c}': df[c].values for c in FEATURES_EMBED}
    data['inp_dense'] = df[FEATURES_DENSE].values
    return data

In [ ]:
#                                                    
# TRAINING LOOP
#                                                    

Folds = 5
skf = StratifiedKFold(n_splits=Folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

X_test_dict = format_data(test_df)

print(f"\n{'='*50}")
print("Training: KERAS DNN with ENTITY EMBEDDINGS")
print(f"{'='*50}")

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df[TARGET]), 1):
    print(f"\n--- Fold {fold} ---")
    
    X_tr = format_data(train_df.iloc[tr_idx])
    y_tr = train_df[TARGET].iloc[tr_idx].values
    
    X_va = format_data(train_df.iloc[va_idx])
    y_va = train_df[TARGET].iloc[va_idx].values
    
    model = build_model(train_df)
    
    # Callbacks
    early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.2, patience=4, min_lr=1e-6)
    
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=50,
        batch_size=1024,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    vp = model.predict(X_va, batch_size=4096).flatten()
    tp = model.predict(X_test_dict, batch_size=4096).flatten()
    
    auc = roc_auc_score(y_va, vp)
    print(f"  Fold {fold} Best AUC = {auc:.6f}")
    
    oof_preds[va_idx] = vp
    test_preds += tp / Folds
    
    tf.keras.backend.clear_session()
    gc.collect()

oof_auc = roc_auc_score(train_df[TARGET], oof_preds)
print(f"\n  >>> DNN OOF AUC: {oof_auc:.6f}")

In [ ]:
# Generate Submission
sub = pd.DataFrame({
    'id': test_ids,
    'Will_Buy_EV': test_preds
})
sub.to_csv('submission.csv', index=False)
print("Submission saved!")
print(sub.head())

---

###   My Top S6E9 Work  

| Notebook | Score | Description |
|:---------|:-----:|:------------|
| [Dual-Objective LGBM (LogLoss + Focal)](https://www.kaggle.com/code/amanatar/s6e9-dual-objective-lgbm-logloss-focal) | **0.94615** | CPU-only, dual loss custom model |
| [Standalone XGBoost (Hist)](https://www.kaggle.com/code/amanatar/s6e9-standalone-xgboost-hist) | **0.94610** | Depth-wise tree growth approach |
| [Standalone CatBoost (Native Cat)](https://www.kaggle.com/code/amanatar/s6e9-standalone-catboost-native-categoricals) |   | Highly robust native encoding |
| [Standalone HistGradientBoosting](https://www.kaggle.com/code/amanatar/s6e9-standalone-histgradientboosting) |   | Ultra-fast Sklearn implementation |
| [Quantum Rank Ensembling](https://www.kaggle.com/code/amanatar/s6e9-quantum-rank-ensembling) | **0.94643+** | Smart blend of top public outputs |

---
*If you find this neural network architecture insightful for tabular data, please drop an Upvote!*  